# 골짜기 급반등 사전 특징 분석

## tl;dr

- 급반등 7건의 장중 최종 저점은 평균 -6.45%, 중앙값 -5.46%, 범위 -4.29%~-10.53%였다.
- 대조군 70건의 평균 급락률(-6.93%)이 오히려 더 깊어, 급락률 하나만으로 급반등을 구분하기 어렵다.
- 급반등군은 같은 날 후보들보다 시가총액이 작고, 전일 거래대금이 직전 5일 중앙값 대비 크게 늘었으며, 최근 5일 거래대금이 상승일에 집중됐다.
- 기관·외국인 실제 순매수는 더 강하지 않았다. 따라서 현재 증거만으로 '개미털기' 또는 주도세력 매집이라고 단정하지 않는다.

## Context & Methods

### Key Assumptions

- 대상: 2026-07-23~24, 09:00~09:20에 전일종가 대비 -4% 이하로 무장한 77건.
- 급반등: 당일 장초반 최종 저점 뒤 10초 이내 +0.6% 이상 반등한 7건.
- 대조군: 같은 정의로 무장했지만 급반등하지 않은 70건.
- 모든 일봉·기관·외국인 자료는 각 사건일 전 날짜만 사용한다.
- 하루별 후보 수 차이(10건 대 67건)를 줄이기 위해 같은 날 내부 백분위와 같은 날 쌍 비교를 사용한다.
- 수수료·세금·스프레드·슬리피지는 이번 사전 특징 분류에는 적용하지 않았다. 손익 백테스트가 아니기 때문이다.

## Data

- 사건 코호트: `analysis/골짜기_급반등_진입비교.csv`
- 급반등 판정: `analysis/골짜기_급반등_FAST_누락진단.log`
- 일봉: `data/eod_daily_bars.csv`
- 실제 기관·외국인 순매수 수량: `data/investor_daily.csv`
- 계산 스크립트: `analysis/골짜기_급반등_사전특징_분석.py`

In [ ]:
from pathlib import Path
import csv
import json

ROOT = Path(r"C:\stock_bot")
DETAIL = ROOT / "analysis" / "골짜기_급반등_사전특징_77종목.csv"
COMPARE = ROOT / "analysis" / "골짜기_급반등_사전특징_비교.csv"
RULES = ROOT / "analysis" / "골짜기_급반등_사전특징_조건후보.csv"
SUMMARY = ROOT / "analysis" / "골짜기_급반등_사전특징_요약.json"

with DETAIL.open(encoding="utf-8-sig", newline="") as handle:
    detail = list(csv.DictReader(handle))
with COMPARE.open(encoding="utf-8-sig", newline="") as handle:
    comparison = list(csv.DictReader(handle))
with RULES.open(encoding="utf-8-sig", newline="") as handle:
    rules = list(csv.DictReader(handle))
summary = json.loads(SUMMARY.read_text(encoding="utf-8"))

assert len(detail) == 77
assert len({(row["day"], row["code"]) for row in detail}) == 77
assert sum(row["quick_v"] == "True" for row in detail) == 7
assert all(int(row["d1_date"]) < int(row["day"]) for row in detail)
print("cohort and no-lookahead checks: PASS")

## Results

### 1. 급락률

In [ ]:
drop = summary["morning_drop"]
print({
    "급반등 평균": round(drop["quick_mean_pct"], 2),
    "급반등 중앙값": round(drop["quick_median_pct"], 2),
    "급반등 범위": (round(drop["quick_min_pct"], 2), round(drop["quick_max_pct"], 2)),
    "대조군 평균": round(drop["control_mean_pct"], 2),
    "대조군 중앙값": round(drop["control_median_pct"], 2),
})

### 2. 전일·최근 5일 특징

In [ ]:
focus = {
    "market_cap_eok",
    "d1_value_vs_prior5",
    "d1_range_pct",
    "d1_close_position",
    "d1_upper_wick_share",
    "signed_value_balance_5d",
    "supply_positive_days_5d",
}
for row in comparison:
    if row["feature"] in focus:
        print(
            row["label"],
            "| 급반등 중앙값:", round(float(row["quick_median"]), 4),
            "| 대조군 중앙값:", round(float(row["control_median"]), 4),
            "| 같은 날 효과:", round(float(row["same_day_pair_effect"]), 3),
        )

### 3. 관찰 우선조건 후보

아래 조건은 같은 표본에서 찾은 탐색 결과다. 실전 매수 조건이 아니라 주문 0의 SHADOW 우선순위로만 검증해야 한다.

In [ ]:
for row in rules[:5]:
    print({
        "조건": row["label"],
        "방향": row["direction"],
        "당일 백분위": float(row["day_percentile_threshold"]),
        "선택": int(row["selected"]),
        "급반등 적중": int(row["quick_hits"]),
        "정밀도%": round(float(row["precision_pct"]), 2),
        "재현율%": round(float(row["recall_pct"]), 2),
    })

## Takeaways

1. -4% 무장선은 유지하는 편이 맞다. 평균 -6.45%로 기준을 낮추면 -4.29%, -4.43%, -5.26%, -5.46% 사례를 놓친다.
2. 전일 사전조건은 매수 허가가 아니라 감시 우선순위에 사용한다.
3. 1차 SHADOW 후보는 같은 날 후보 중 시총 하위 25%, 전일 거래대금/직전5일 중앙값 상위 25%, 최근5일 방향성 거래대금 균형 상위 25% 중 2개 이상이다.
4. 이 2-of-3 후보는 동일 표본에서 17건을 골라 급반등 4건을 포함했다(정밀도 23.5%, 재현율 57.1%). 독립 거래일 재검증 전에는 실전 필터로 쓰지 않는다.
5. 실제 진입은 별도의 초단기 저점·매수/매도 체결 우위 확인으로 결정해야 한다.